In [1]:
import pandas as pd
from matplotlib import pyplot as plt 
from matplotlib.gridspec import GridSpec
from matplotlib.transforms import Bbox
import re

import glob
import numpy as np

import io
from matplotlib import colormaps
from matplotlib.colors import ListedColormap, BoundaryNorm
from Bio import pairwise2

from Bio import Align



/vol/ek/Home/tomertsa/miniforge3/envs/local_python_playground/lib/python3.12/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [3]:
def detect_reliable_alignment_patterns(pattern_str):
    """
    Detect specific patterns in alignment string while filtering out unreliable regions
    with too many dots in the surrounding context.
    
    Args:
        pattern_str: String containing only '.', '-', or '|' characters
        
    Returns:
        dict: Dictionary with boolean values indicating if each reliable pattern was found
            and the matching substrings if found
    """
    results = {
        'pattern1': {
            'found': False,
            'matches': []
        },
        'pattern2': {
            'found': False,
            'matches': []
        }
    }
    
    # Base patterns to match 
    
    pattern1 = r'\|{5,}\.?-{5,}\.?\|{5,}'
    pattern2 = r'-{5,}\.?\|{5,}'
    
    # Find all occurrences of pattern 1
    matches1 = re.finditer(pattern1, pattern_str)
    
    for match in matches1:

        results['pattern1']['found'] = True
        results['pattern1']['matches'].append({
            'substring': match.group(),
            'position': match.span(),
            'context': {
                'left': pattern_str[max(0, match.start() - 10):match.start()],
                'right': pattern_str[match.end():min(len(pattern_str), match.end() + 10)]
            }
        })

    # Find all occurrences of pattern 2
    matches2 = re.finditer(pattern2, pattern_str)
    print(pattern_str)
    for match in matches2:
        print(match.span())
        results['pattern2']['found'] = True
        results['pattern2']['matches'].append({
            'substring': match.group(),
            'position': match.span(),
            'context': {
                'left': pattern_str[max(0, match.start() - 10):match.start()],
                'right': pattern_str[match.end():min(len(pattern_str), match.end() + 10)]
            }
        })
    
    return results


def extract_alignment_pattern(alignment):
    """
    Extract the alignment pattern from a BioPython PairwiseAlignment object.
    Only returns characters '.', '-', or '|'.
    
    Args:
        alignment: A BioPython PairwiseAlignment object
        
    Returns:
        str: A concatenated string of the alignment pattern containing only '.', '-', or '|'
    """
#     print("STR ALIGN")
#     print(repr(str(alignment)))
    
    lines = str(alignment).split('\n')
    
    # Filter for alignment rows (exclude empty lines and lines starting with 'target' or 'query')
    align_rows = [
        line for line in lines 
        if line.strip() and 
        not line.strip().startswith(('target', 'query')) 
    ]
    align_str = "".join(align_rows)
    parsed_string = ''.join(c for c in align_str if c in {'.', '-', '|'})
#     print('align_rows')
#     print(align_rows)
#     print('align_str')
#     print(align_str)
#     print('PARSED STRING')
#     print(parsed_string)
#     print(parsed_string)
    return parsed_string 


def get_class_from_alignment(mane_id, iso_id, alignment):
    target_seq = alignment[0].target
    query_seq = alignment[0].query
    
    if target_seq == query_seq:
        return "Identical"
        
    elif alignment[0].counts()[0] == 0:
        return "no_gaps_full_align"

    # take the x-most nterminal positions of the alignment, and serach for full gaps 
    align_pattern = extract_alignment_pattern(alignment[0])
#     print(align_pattern)
    
    detected_patterns = detect_reliable_alignment_patterns(align_pattern)
    print(detected_patterns['pattern1']['matches'])
    print(detected_patterns['pattern2']['matches'])

    
    if detected_patterns['pattern1']['found'] == True and detected_patterns['pattern1']['matches'][0]['position'][0] == 0:
        return 'aligned_sp_and_deletion'
#     if detected_patterns['pattern2']['found'] == False:
#         return 'other'
    elif detected_patterns['pattern2']['found'] == True and detected_patterns['pattern2']['matches'][0]['position'][0] == 0:
        return 'n_terminal_truncation'
    else:
        return 'other'


def pairwise_align(seq1, seq2):
    aligner = Align.PairwiseAligner()
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.1
    aligner.target_end_gap_score = 0.0
    aligner.query_end_gap_score = 0.0

    alignments = aligner.align(seq1, seq2)
    
    return alignments
    
def match_mane_iso_and_align(iso_id):
    
    chs_group_i = ".".join(iso_id.split('.')[0:2])
    group_df = chs_mane_signalp.query('CHESS_group == @chs_group_i')
    group_mane_ids = group_df.query('MANE_status == "mane"').CHESS_ID
    if len(group_mane_ids) == 0:
        return ['no_mane',iso_id,0]
    else:    

        for mane_i in group_mane_ids:
            print("MANE "+mane_i)
            print("ISO " +iso_id)
            mane_i_seq = chs_fasta_table.query('CHESS_ID == @mane_i').Sequence.iloc[0]
            iso_seq = chs_fasta_table.query('CHESS_ID == @iso_id').Sequence.iloc[0]
            alignments_i = pairwise_align(mane_i_seq, iso_seq)
            class_i = get_class_from_alignment(mane_i, iso_id, alignments_i)
            print(class_i)

            print(alignments_i[0])
    #         print(str(alignments_i[0]).split('\n')[1].split(' '))

            # write alignment files out as txt 
            write_bio_alignment_to_file(mane_i, iso_id, alignments_i, class_i)

            output = [mane_i, iso_id, class_i]
            print('output is ')
            print(output)
            return output
    
    
def write_bio_alignment_to_file(mane_id, iso_id, alignment, class_i):
    """
    Write Biopython alignment output to a text file in the same format as it prints.
    
    Args:
        score (float): Initial alignment score
        mane_id (str): MANE identifier
        iso_id (str): ISO identifier
        alignment (Bio.Align.PairwiseAlignment): Biopython alignment object
        final_score (float): Final score
        output_file (str): Path to output file
    """
    output_file = 'pairwise_align_'+'mane_'+str(mane_id)+'_iso_'+str(iso_id)+'_'+str(class_i)+'.txt'
    with open(output_file, 'w') as f:
        # Write header information
        f.write(f"MANE {mane_id}\n")
        f.write(f"ISO {iso_id}\n")
        i_score = alignment[0].score
        i_length = alignment[0].length
        length_norm_score = i_score / i_length 
        
        f.write(f"Class {class_i}\n")             
        f.write(f"Score {i_score}\n")
        f.write(f"Aln length {i_length}\n")
        f.write(f"Score divided by length {length_norm_score}\n")
        
        # Write the alignment - get the string representation
        alignment_str = str(alignment[0])
        f.write(alignment_str)
        
        


In [4]:
# read in the full table of sasa_lddt_nterm_avgs 
sasa_lddt_avg_tab = pd.read_table('results/merged_sp_inclusive_sasa_lddt_avg_nterm_post_nterm_by_signalp_cs_pred.tsv',sep='\t')
# sasa_lddt_avg_tab.set_index('id',inplace=True)

In [5]:
chs_mane_signalp = pd.read_table('results/chs_mane_signalp_241024.tsv', sep='\t')
all_chs_iso = pd.read_table('results/all_chs3_isoforms_with_MANE_select_annotation_241024.tsv',sep='\t')

# format the mane_status column 
chs_mane_signalp.MANE_status.replace('MANE Select', 'mane', inplace=True)
chs_mane_signalp.MANE_status.replace('MANE Plus Clinical', 'mane', inplace=True)
chs_mane_signalp.MANE_status.fillna('isoform', inplace=True)


# create a chess_group column for easier detection of mane's 
chs_mane_signalp['CHESS_group'] = chs_mane_signalp.CHESS_ID.apply(lambda x: ".".join(x.split('.')[0:2]))

/tmp/ipykernel_990829/831511155.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  chs_mane_signalp.MANE_status.replace('MANE Select', 'mane', inplace=True)
/tmp/ipykernel_990829/831511155.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [6]:
# read in the chs fasta table and change CHS_ID to index 
chs_fasta_table = pd.read_table('results/chs_fasta_table.tsv',sep='\t')
chs_fasta_table.set_index('CHESS_ID',inplace=True)

In [6]:
# read in the general info tab of inclusive sp isoforms 
inclusive_sp_general_tab = pd.read_table('results/inclusive_sp_general_tab_120125_struct_role_domain_completion_ipynb.tsv')

In [8]:
# filter structural candidates for seq align using the relevant filters 
candidate_tab = sasa_lddt_avg_tab.query('nterm_avg_sasa <= 90')

In [98]:
# # collect the classes of each seq align pair, and create a df 

# class_ser = candidate_tab.id.apply(lambda x: match_mane_iso_and_align(x))                 

# class_tab = pd.DataFrame(class_ser.dropna().tolist(), columns = ['aligned_mane', 'aligned_iso', 'aligned_class'])

# class_tab.to_csv('results/classification_of_seq_align_struct_candidates_isoform_mane_table_220225.tsv',sep='\t',index=False)

MANE CHS.30918.3
ISO CHS.30918.3
Identical
target            0 MELSWHVVFIALLSFSCWGSDWESDRNFISTAGPLTNDLLHNLSGLLGDQSSNFVAGDKD
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MELSWHVVFIALLSFSCWGSDWESDRNFISTAGPLTNDLLHNLSGLLGDQSSNFVAGDKD

target           60 MYVCHQPLPTFLPEYFSSLHASQITHYKVFLSWAQLLPAGSTQNPDEKTVQCYRRLLKAL
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 MYVCHQPLPTFLPEYFSSLHASQITHYKVFLSWAQLLPAGSTQNPDEKTVQCYRRLLKAL

target          120 KTARLQPMVILHHQTLPASTLRRTEAFADLFADYATFAFHSFGDLVGIWFTFSDLEEVIK
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           120 KTARLQPMVILHHQTLPASTLRRTEAFADLFADYATFAFHSFGDLVGIWFTFSDLEEVIK

target          180 ELPHQESRASQLQTLSDAHRKAYEIYHESYAFQGGKLSVVLRAEDIPELLLEPPISALAQ
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           180 ELPHQESRASQLQTLSDAHRKAYEIYHESYAFQGGKLSVVLRA

MANE CHS.5606.19
ISO CHS.5606.alt117
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||---------------------------||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
(0, 266)
(266, 445)
[{'substring': '|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||---------------------------||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||', 'position': (173, 445), 'context': {'left': '----------', 'right': ''}}]
[{'substring': '---------------------------------------------------------------------------------------------------------------------

MANE CHS.35724.4
ISO CHS.35724.alt111
Identical
target            0 MEGAALLRVSVLCIWMSALFLGVGVRAEEAGARVQQNVPSGTDTGDPQSKPLGDWAAGTM
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MEGAALLRVSVLCIWMSALFLGVGVRAEEAGARVQQNVPSGTDTGDPQSKPLGDWAAGTM

target           60 DPESSIFIEDAIKYFKEKVSTQNLLLLLTDNEAWNGFVAAAELPRNEADELRKALDNLAR
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 DPESSIFIEDAIKYFKEKVSTQNLLLLLTDNEAWNGFVAAAELPRNEADELRKALDNLAR

target          120 QMIMKDKNWHDKGQQYRNWFLKEFPRLKSELEDNIRRLRALADGVQKVHKGTTIANVVSG
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           120 QMIMKDKNWHDKGQQYRNWFLKEFPRLKSELEDNIRRLRALADGVQKVHKGTTIANVVSG

target          180 SLSISSGILTLVGMGLAPFTEGGSLVLLEPGMELGITAALTGITSSTMDYGKKWWTQAQA
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           180 SLSISSGILTLVGMGLAPFTEGGSLVLLEPGMELGITA

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
(0, 955)
[]
[{'substring': '----------------

MANE CHS.53322.4
ISO CHS.53322.alt106
Identical
target            0 MSGAGRALAALLLAASVLSAALLAPGGSSGRDAQAAPPRDLDKKRHAELKMDQALLLIHN
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MSGAGRALAALLLAASVLSAALLAPGGSSGRDAQAAPPRDLDKKRHAELKMDQALLLIHN

target           60 ELLWTNLTVYWKSECCYHCLFQVLVNVPQSPKAGKPSAAAASVSTQHGSILQLNDTLEEK
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 ELLWTNLTVYWKSECCYHCLFQVLVNVPQSPKAGKPSAAAASVSTQHGSILQLNDTLEEK

target          120 EVCRLEYRFGEFGNYSLLVKNIHNGVSEIACDLAVNEDPVDSNLPVSIAFLIGLAVIIVI
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           120 EVCRLEYRFGEFGNYSLLVKNIHNGVSEIACDLAVNEDPVDSNLPVSIAFLIGLAVIIVI

target          180 SFLRLLLSLDDFNNWISKAISSRETDRLINSELGSPSRTDPLDGDVQPATWRLSALPPRL
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query           180 SFLRLLLSLDDFNNWISKAISSRETDRLINSELGSPSR

...

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------...|.......||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||.....................-----------------------------------------

|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||.............|..|......|...|......-----------------------------------------
[]
[]
other
target            0 MAGKGSSGRRPLLLGLLVAVATVHLVICPYTKVEESFNLQATHDLLYHWQDLEQYDHLEF
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MAGKGSSGRRPLLLGLLVAVATVHLVICPYTKVEESFNLQATHDLLYHWQDLEQYDHLEF

target           60 PGVVPRTFLGPVVIAVFSSPAVYVLSLLEMSKFYSQLIVRGVLGLGVIFGLWTLQKEVRR
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query            60 PGVVPRTFLGPVVIAVFSSPAVYVLSLLEMSKFYSQLIVRGVLGLGVIFGLWTLQKEVRR

target     

-------------------------------------------------------------------------------------..............|.||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||----------------------------------------------------------
[]
[]
other
target            0 MEGNRDEAEKCVEIAREALNAGNREKAQRFLQKAEKLYPLPSARALLEIIMKNGSTAGNS
                  0 ------------------------------------------------------------
query             0 ------------------------------------------------------------

target           60 PHCRKPSGSGDQSKPNCTKDSTSGSGEGGKGYTKDQVDGVLSINKCKNYYEVLGVTKDAG
                 60 -------------------------..............|.|||||||||||||||||||
query             0 -------------------------MTVFLEIYIITHSLVHSINKCKNYYEVLGVTKDAG

target          120 DEDLKKAYRKLALKFHPDKNHAPGATDAFKKIGNAYAVLSNPEKRKQYDLTGNEEQACNH
                120 |||||||||||||||||||

MANE CHS.642.4
ISO CHS.642.3
||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||....--------------------------------------------------------------
[]
[]
other
target            0 MSESELGRKWDRCLADAVVKIGTGFGLGIVFSLTFFKRRMWPLAFGSGMGLGMAYSNCQH
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
query             0 MSESELGRKWDRCLADAVVKIGTGFGLGIVFSLTFFKRRMWPLAFGSGMGLGMAYSNCQH

target           60 DFQAPYLLHGKYVKEQEQ------------------------------------------
                 60 ||||||||||||||....------------------------------------------
query            60 DFQAPYLLHGKYVKNLLKPSAACSQPMSAGEMGVGEPSLHQSKSEVITGTMQEEAQLCGT

target           78 --------------------  78
                120 -------------------- 140
query           120 EAGSRPFPPLPGSSLSWHQS 140

output is 
['CHS.642.4', 'CHS.642.3', 'other']
MANE CHS.13810.2
ISO CHS.13810.alt100
||||||||||||||||||||||-------------------------------------------------------------------------||||||||||||||

---------------------------------------------------....||......|..........|....|......................................|.....|...|||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
[]
[]
other
target            0 MASEFKKKLFWRAVVAEFLATTLFVFISIGSALGFKYPVGNNQTAVQDNVKVSLAFGLSI
                  0 ---------------------------------------------------....||...
query             0 ---------------------------------------------------MFWTFGYEA

target           60 ATLAQSVGHISGAHLNPAVTLGLLLSCQISIFRALMYIIAQCVGAIVATAILSGITSSLT
                 60 ...|..........|....|......................................|.
query             9 VSPAGPSHLFASLLLGVLLTITFMPGARPLPLVLVPQNTLAWMQLDAKAPAHPRPLQLLG

target          120 GNSLGRNDLADGVNSGQGLGIEIIGTLQLVLCVLATTDRRRRDLGGSAPLAIGLSVALGH
                120 ....|...||||||||||||||||||||||||||||||||||||||||||||||||||||
query            69 RVGPGSRQLADGVNSGQGLGIEIIGTLQLVLCVLATTDRRRRDLGGSA

In [3]:
class_tab = pd.read_table('results/classification_of_seq_align_struct_candidates_isoform_mane_table_220225.tsv',sep='\t')

In [4]:
# '0' stands for - no MANE annotated for isoform (in match_mane_iso_and_align function)
class_tab.aligned_class.unique()

array(['Identical', 'n_terminal_truncation', 'aligned_sp_and_deletion',
       '0', 'other'], dtype=object)

In [6]:
# filter out cases where "no mane" is annotated for the chess group 
class_tab.aligned_class.value_counts()

aligned_class
n_terminal_truncation      713
other                      424
Identical                  278
aligned_sp_and_deletion    124
0                           59
Name: count, dtype: int64

In [8]:
# check for redundant isoforms - matched to more than 1 mane 
len(class_tab.aligned_iso.unique())

NameError: name 'class_tab' is not defined

In [124]:
# validate no duplications 
# keep only the aligned_sp_and_deletion isoforms 
sp_and_del = class_tab.query('aligned_class == "aligned_sp_and_deletion"')
sp_and_del['CHESS_group'] =  sp_and_del.aligned_iso.apply(lambda x:".".join(x.split('.')[0:2]))

print(len(sp_and_del.query('aligned_class == "aligned_sp_and_deletion"').aligned_iso.unique()))
print(len(sp_and_del.query('aligned_class == "aligned_sp_and_deletion"').aligned_mane.unique()))
print(len(sp_and_del.query('aligned_class == "aligned_sp_and_deletion"').CHESS_group.unique()))

124
71
71


/tmp/ipykernel_3270258/2632564871.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sp_and_del['CHESS_group'] =  sp_and_del.aligned_iso.apply(lambda x:".".join(x.split('.')[0:2]))


In [126]:
# save aligned_sp_and_deletion table out separately

sp_and_del.to_csv('results/aligned_sp_and_deletion_isoform_mane_table_220225.tsv',sep='\t',index=False)